# AutoSemilla VQCC — interfaz local

Panel de trabajo para ejecutar el motor de búsqueda en tu equipo, revisar el historial, exportar el punto de control e ingerir resultados de Kickmix.

**Requisitos:** Python 3.10+ y Jupyter (`pip install notebook`). Abre el notebook desde la carpeta del proyecto con `jupyter notebook`.

**Aviso:** la puntuación proxy NO es la puntuación oficial de la competición. Solo los resultados verificados por el verificador oficial cuentan como verificados.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from autosemilla.core import Ledger, AdaptiveSearch, export_jsonl
from autosemilla.kickmix_ingest import ingest_kickmix_jsonl

STATE = ROOT / "state"
DB_PATH = STATE / "ledger.sqlite"
EXPORT_PATH = STATE / "experiments.jsonl"
KICKMIX_RESULTS = STATE / "kickmix_results.jsonl"

ledger = Ledger(DB_PATH)
print("Historial local:", DB_PATH)

## 1. Configuración de la búsqueda

Cambia estos valores y vuelve a ejecutar la siguiente celda.

In [ ]:
BENCHMARK = "invsqrt"      # "invsqrt", "dqi" o "dfthcsos"
N_CANDIDATES = 64         # número de candidatos a generar (1-10000)
EXPLORATION = 0.30        # 0 = solo explotar, 1 = solo explorar

## 2. Ejecutar búsqueda

In [ ]:
def tabla(encabezados, filas):
    lineas = ["| " + " | ".join(encabezados) + " |", "|" + "---|" * len(encabezados)]
    lineas += ["| " + " | ".join(str(c) for c in f) + " |" for f in filas]
    return "\n".join(lineas)

search = AdaptiveSearch(ledger, exploration=EXPLORATION)
rows = search.run(BENCHMARK, N_CANDIDATES)
promoted = search.promote(BENCHMARK, 5)

display(Markdown(
    f"**Evaluados:** {len(rows)} · **Promovidos:** {len(promoted)} · "
    f"**Historial ({BENCHMARK}):** {ledger.count(BENCHMARK)} · "
    f"**Verificados oficiales:** {ledger.verified_count(BENCHMARK)}"
))
display(Markdown("### Mejores candidatos de esta ejecución (proxy)\n\n" + tabla(
    ["ID", "Puntuación proxy", "Parámetros"],
    [(s.id, round(e.cheap_score, 3), s.params) for s, e in rows[:5]],
)))

## 3. Historial del benchmark

In [ ]:
historial = ledger.top(BENCHMARK, 10)
display(Markdown(tabla(
    ["ID", "Puntuación proxy", "Estado", "Verificado oficialmente"],
    [(r["seed_id"], round(r["cheap_score"], 3), r["promotion"], "Sí" if r["official_verified"] else "No") for r in historial],
)) if historial else "Todavía no hay candidatos válidos para este benchmark.")

## 4. Ingerir resultados de Kickmix

Lee `state/kickmix_results.jsonl` si existe. Solo se aceptan resultados que pasen el verificador oficial.

In [ ]:
if KICKMIX_RESULTS.exists():
    counts = ingest_kickmix_jsonl(ledger, KICKMIX_RESULTS)
    print("Resultados ingeridos:", counts)
else:
    print(f"No hay resultados de Kickmix en {KICKMIX_RESULTS}.")

## 5. Exportar punto de control

In [ ]:
registros = export_jsonl(ledger, EXPORT_PATH)
print(f"Exportados {registros} registros a {EXPORT_PATH}")